# 多期 Brinson 归因与链接

本教程使用同一份本地模拟 Dow 30 组合的 12 个月数据。凡涉及多期链接效应的图表与表格，都标明所选 linking 方法。收益率与效应采用十进制。

## 1. 读取本地数据

CSV 输入位于 `data/dow30_brinson/`。可从项目根目录或任一语言教程目录启动 Jupyter。

In [ ]:
from pathlib import Path
import sys
import polars as pl
import matplotlib.pyplot as plt

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "brinsons").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Launch Jupyter from the project root or tutorials/.")
sys.path.insert(0, str(ROOT / "src"))
from brinsons import (
    ColumnNames, bhb, fachler, calculate_multiple_period,
    link_carino, link_grap, link_frongello, link_menchero, link_geometric,
    print_multiple_period_category_weights, print_multiple_period_category_returns,
    print_multiple_period_category_attribution, print_multiple_period_total_attribution,
    print_multiple_period_attribution_summary, chart_multiple_period_category_returns,
    chart_multiple_period_category_weights, chart_multiple_period_category_attribution,
    chart_multiple_period_excess_return_waterfall,
)

asset = pl.read_csv(ROOT / "data" / "dow30_brinson" / "asset_table.csv")
periods = pl.read_csv(ROOT / "data" / "dow30_brinson" / "periods.csv")
print(f"{asset['Name'].n_unique()} stocks, {periods.height} monthly periods")
display(asset.head(6))

## 2. 汇总行业收益

按月和行业汇总个股贡献。`ColumnNames` 保留原始输入列名。

In [ ]:
sector = asset.group_by(["Period", "Category"]).agg(
    pl.col("PortfolioWeight", "BenchmarkWeight").sum(),
    (pl.col("PortfolioWeight") * pl.col("Return")).sum().alias("PortfolioContribution"),
    (pl.col("BenchmarkWeight") * pl.col("Return")).sum().alias("BenchmarkContribution"),
).with_columns(
    (pl.col("PortfolioContribution") / pl.col("PortfolioWeight")).alias("PortfolioReturn"),
    (pl.col("BenchmarkContribution") / pl.col("BenchmarkWeight")).alias("BenchmarkReturn"),
).sort(["Period", "Category"])
column_names = ColumnNames(period="Period", segment="Category",
    portfolio_weight="PortfolioWeight", benchmark_weight="BenchmarkWeight",
    portfolio_return="PortfolioReturn", benchmark_return="BenchmarkReturn")
display(sector.head())

## 3. 计算各期效应并链接

`calculate_multiple_period` 分别返回各期和链接后的结果。这里的 Brinson 方法为 BF，linking 方法为 **Menchero**。算术链接目标是 $\prod_t(1+R_{P,t})-\prod_t(1+R_{B,t})$。

In [ ]:
multiple = calculate_multiple_period(sector, attribution=fachler, linking=link_menchero, columns=column_names)
print('Multiple-period linking method: Menchero')
result = multiple.periods  # reporting tools link the period results with Menchero
display(multiple.periods.summary.head())
display(multiple.linked.summary)

## 4. 在同一数据上比较所有 linking 方法

Cariño、GRAP、Frongello、Menchero 对账到复合后收益之差；Geometric 对账到相对复合收益，并含乘法交叉项。同时比较 BHB/BF 与两效应/三效应。

In [ ]:
linkers = [("Cariño", link_carino), ("GRAP", link_grap),
           ("Frongello", link_frongello), ("Menchero", link_menchero),
           ("Geometric", link_geometric)]
arithmetic_rows = []
geometric_rows = []
for brinson_name, brinson_fn in [("BHB", bhb), ("BF", fachler)]:
    for effect_count in (2, 3):
        for linker_name, linker_fn in linkers:
            run = calculate_multiple_period(sector, attribution=brinson_fn,
                linking=linker_fn, effects=effect_count, columns=column_names)
            names = run.linked.columns
            totals = run.linked.summary.row(0, named=True)
            allocation = totals[names.summary_allocation]
            selection = totals[names.summary_selection]
            interaction = totals[names.summary_interaction]
            active = totals[names.summary_excess_return]
            common = {"Brinson": brinson_name, "Effects": effect_count, "Linking": linker_name,
                      "Portfolio (bp)": 10_000 * totals[names.summary_portfolio_return],
                      "Benchmark (bp)": 10_000 * totals[names.summary_benchmark_return]}
            if linker_name == "Geometric":
                geometric_rows.append({**common, "Relative active (bp)": 10_000 * active,
                    "Allocation (bp)": 10_000 * allocation,
                    "Selection incl. interaction (bp)": 10_000 * selection,
                    "Cross term (bp)": 10_000 * allocation * selection,
                    "Residual (bp)": 10_000 * ((1 + allocation) * (1 + selection) - 1 - active)})
            else:
                arithmetic_rows.append({**common, "Active (bp)": 10_000 * active,
                    "Allocation (bp)": 10_000 * allocation,
                    "Selection (bp)": 10_000 * selection,
                    "Interaction (bp)": 10_000 * interaction,
                    "Residual (bp)": 10_000 * (allocation + selection + interaction - active)})

arithmetic_comparison = pl.DataFrame(arithmetic_rows)
geometric_comparison = pl.DataFrame(geometric_rows)
assert arithmetic_comparison["Residual (bp)"].abs().max() < 1e-6
assert geometric_comparison["Residual (bp)"].abs().max() < 1e-6
with pl.Config(tbl_rows=100, tbl_cols=12, tbl_width_chars=180, fmt_str_lengths=32):
    display(arithmetic_comparison)
    display(geometric_comparison)


### 逐一显示每种 linking 方法的总归因

In [ ]:
for linker_name, linker_fn in linkers:
    print_multiple_period_total_attribution(result, linking=linker_fn)


### 算术 linking 对比图

每行都标明所用方法。本实现中 GRAP 与 Frongello 使用相同链接系数。

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10), sharex=True, layout="constrained")
for ax, (brinson_name, effect_count) in zip(axes.flat,
                                            [("BHB", 2), ("BHB", 3), ("BF", 2), ("BF", 3)]):
    rows = arithmetic_comparison.filter(
        (pl.col("Brinson") == brinson_name) & (pl.col("Effects") == effect_count))
    positions = list(range(rows.height))
    for offset, field, color in [(-0.23, "Allocation (bp)", "#2874a6"),
                                 (0.00, "Selection (bp)", "#d88935"),
                                 (0.23, "Interaction (bp)", "#5b8a5a")]:
        ax.barh([p + offset for p in positions], rows[field].to_list(),
                height=0.21, label=field.replace(" (bp)", ""), color=color)
    ax.set_yticks(positions, rows["Linking"].to_list())
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title(f"{brinson_name}, {effect_count} effects")
    ax.grid(axis="x", alpha=0.2)
axes[0, 0].legend(loc="lower right", fontsize=8)
for ax in axes[1]:
    ax.set_xlabel("Full-horizon attributed effect (bp)")
fig.suptitle("Linking: Cariño / GRAP / Frongello / Menchero — four Brinson settings")
plt.show()


### 几何 linking 对比图

配置加选股通常不等于相对总收益；还需计入两者的交叉项。

In [ ]:
geometric_labels = [f"{row['Brinson']} · {row['Effects']} effects"
                    for row in geometric_comparison.iter_rows(named=True)]
positions = list(range(geometric_comparison.height))
fig, (ax_large, ax_small) = plt.subplots(1, 2, figsize=(14, 5), sharey=True, layout="constrained")
for offset, field, color, label in [
    (-0.15, "Allocation (bp)", "#2874a6", "Allocation"),
    (0.15, "Selection incl. interaction (bp)", "#d88935", "Selection incl. interaction")]:
    ax_large.barh([p + offset for p in positions], geometric_comparison[field].to_list(),
                  height=0.26, label=label, color=color)
for offset, field, color, label in [
    (-0.15, "Cross term (bp)", "#5b8a5a", "Allocation × selection"),
    (0.15, "Relative active (bp)", "#713f8e", "Relative active total")]:
    ax_small.barh([p + offset for p in positions], geometric_comparison[field].to_list(),
                  height=0.26, label=label, color=color)
ax_large.set_yticks(positions, geometric_labels)
for axis in (ax_large, ax_small):
    axis.axvline(0, color="black", linewidth=0.8)
    axis.grid(axis="x", alpha=0.2)
    axis.legend(fontsize=8)
ax_large.set_xlabel("Main geometric effects (bp)")
ax_small.set_xlabel("Cross term and relative total (bp, zoomed)")
fig.suptitle("Linking: Geometric — separate scales for effects and reconciliation")
plt.show()


## 5. 行业权重（无 linking）

In [ ]:
print_multiple_period_category_weights(result);

## 6. 行业收益（无 linking）

In [ ]:
print_multiple_period_category_returns(result);

## 7. 行业归因（Menchero linking）

In [ ]:
print_multiple_period_category_attribution(result, linking=link_menchero);

## 8. 总归因（Menchero linking）

In [ ]:
print_multiple_period_total_attribution(result, linking=link_menchero);

## 9. 绩效归因摘要（Menchero linking）

In [ ]:
print_multiple_period_attribution_summary(result, linking=link_menchero);

## 10. 横向柱状图：行业收益（无 linking）

In [ ]:
fig, ax = chart_multiple_period_category_returns(result)
plt.show()

## 11. 横向柱状图：行业权重（无 linking）

In [ ]:
fig, ax = chart_multiple_period_category_weights(result)
plt.show()

## 12. 横向柱状图：行业归因（Menchero linking）

In [ ]:
fig, ax = chart_multiple_period_category_attribution(result, linking=link_menchero)
plt.show()

## 13. 四种算术 linking 的行业瀑布图

In [ ]:
for linker_name, linker_fn in linkers:
    if linker_fn is link_geometric:
        continue  # geometric linking has no category-level linked effects
    fig, ax = chart_multiple_period_excess_return_waterfall(
        result, by_category=True, linking=linker_fn)
    plt.show()


## 14. 五种 linking 的总体瀑布图

In [ ]:
for linker_name, linker_fn in linkers:
    fig, ax = chart_multiple_period_excess_return_waterfall(
        result, by_category=False, linking=linker_fn)
    plt.show()


行业层链接效应适用于四种算术方法。几何瀑布图显示总体配置、选股、交叉项及相对超额收益。不同 linking 方法可能改变效应分配，但须满足各自的对账恒等式。